## Spark GroupBy and Aggregate Functions

* *Groupby* allows you to group rows together based off some column value.
* Once you have performed the *Groupby* operation you can use an aggregate function off that data.
* An aggregate function aggregates multiple rows of data into a single output, such as taking the sum of inputs, or counting the number of inputs.

In [1]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName("groupbyagg").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 21:06:47 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/03/07 21:06:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 21:06:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Import data

In [2]:
!curl https://raw.githubusercontent.com/markumreed/colab_pyspark/main/sales_data.csv >> sales_data.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   202  100   202    0     0    515      0 --:--:-- --:--:-- --:--:--   515


In [3]:
import pandas as pd

In [7]:
df = spark.read.csv("sales_data.csv", inferSchema=True, header=True)
df.show()

+-------+--------------+---------+
|company|representative|num_sales|
+-------+--------------+---------+
|    XYZ|           Bob|    200.0|
|    XYZ|           Tom|    120.0|
|    XYZ|         Frank|    340.0|
|   ABCD|         Jerry|    600.0|
|   ABCD|           Amy|    124.0|
|   ABCD|       Vanessa|    243.0|
|     OK|          Carl|    870.0|
|     OK|         Sarah|    350.0|
|   BLAH|          John|    250.0|
|   BLAH|         Linda|    130.0|
|   BLAH|          Mike|    750.0|
|   BLAH|         Chris|    350.0|
+-------+--------------+---------+



In [8]:
df.printSchema()

root
 |-- company: string (nullable = true)
 |-- representative: string (nullable = true)
 |-- num_sales: double (nullable = true)



#### Grouping Data

* Group the data by company

In [9]:
df.groupBy("company")

GroupedData[grouping expressions: [company], value: [company: string, representative: string ... 1 more field], type: GroupBy]

#### Aggregate Functions

* mean, count, max, min, sum...

In [10]:
df.groupby("company").mean().show()

+-------+-----------------+
|company|   avg(num_sales)|
+-------+-----------------+
|   BLAH|            370.0|
|    XYZ|            220.0|
|     OK|            610.0|
|   ABCD|322.3333333333333|
+-------+-----------------+



In [11]:
df.groupBy('company').count().show()

+-------+-----+
|company|count|
+-------+-----+
|   BLAH|    4|
|    XYZ|    3|
|     OK|    2|
|   ABCD|    3|
+-------+-----+



In [12]:
df.groupBy('company').min().show()

+-------+--------------+
|company|min(num_sales)|
+-------+--------------+
|   BLAH|         130.0|
|    XYZ|         120.0|
|     OK|         350.0|
|   ABCD|         124.0|
+-------+--------------+



In [13]:
df.groupBy('company').max().show()

+-------+--------------+
|company|max(num_sales)|
+-------+--------------+
|   BLAH|         750.0|
|    XYZ|         340.0|
|     OK|         870.0|
|   ABCD|         600.0|
+-------+--------------+



In [14]:
df.groupBy('company').sum().show()

+-------+--------------+
|company|sum(num_sales)|
+-------+--------------+
|   BLAH|        1480.0|
|    XYZ|         660.0|
|     OK|        1220.0|
|   ABCD|         967.0|
+-------+--------------+



### Aggregating

* Not all methods need a groupby call, instead you can just call the generalized *.agg()* method, that will call the aggregate across all rows in the dataframe column specified.
* It can take in arguments as a single column, or create multiple aggregate calls all at once using dictionary notation.

In [17]:
df.agg({"num_sales": "max", "num_sales": "min"}).show()

+--------------+
|min(num_sales)|
+--------------+
|         120.0|
+--------------+



In [18]:
df.groupBy("company").agg({"num_sales": "mean"}).show()

+-------+-----------------+
|company|   avg(num_sales)|
+-------+-----------------+
|   BLAH|            370.0|
|    XYZ|            220.0|
|     OK|            610.0|
|   ABCD|322.3333333333333|
+-------+-----------------+



In [20]:
company_groups = df.groupBy("company")
company_groups.mean().show()

+-------+-----------------+
|company|   avg(num_sales)|
+-------+-----------------+
|   BLAH|            370.0|
|    XYZ|            220.0|
|     OK|            610.0|
|   ABCD|322.3333333333333|
+-------+-----------------+



### Functions

* There are a variety of functions you can import from *pyspark.sql.functions*.

In [22]:
from pyspark.sql.functions import countDistinct, avg, stddev

In [23]:
df.select(countDistinct("num_sales")).show()

+-------------------------+
|count(DISTINCT num_sales)|
+-------------------------+
|                       11|
+-------------------------+



#### Alias

* To change the name, use the *.alias()* method for this:

In [25]:
df.select(countDistinct('num_sales').alias('Distinct Sales')).show()

+--------------+
|Distinct Sales|
+--------------+
|            11|
+--------------+



In [31]:
df.select(avg("num_sales").alias('Average Sales')).show()

+-----------------+
|    Average Sales|
+-----------------+
|360.5833333333333|
+-----------------+



In [33]:
df.select(stddev("num_sales").alias('Standard deviation of sales')).show()

+---------------------------+
|Standard deviation of sales|
+---------------------------+
|         250.08742410799007|
+---------------------------+



#### Precision

* Use the *format_number* to change precision.

In [29]:
from pyspark.sql.functions import format_number

In [35]:
sales_std = df.select(stddev("num_sales").alias("stddev"))

In [36]:
sales_std.show()

+------------------+
|            stddev|
+------------------+
|250.08742410799007|
+------------------+



In [37]:
sales_std.select(format_number("stddev", 2)).show()

+------------------------+
|format_number(stddev, 2)|
+------------------------+
|                  250.09|
+------------------------+



#### Order By

In [39]:
df.orderBy("num_sales").show()  # Ascending Order

+-------+--------------+---------+
|company|representative|num_sales|
+-------+--------------+---------+
|    XYZ|           Tom|    120.0|
|   ABCD|           Amy|    124.0|
|   BLAH|         Linda|    130.0|
|    XYZ|           Bob|    200.0|
|   ABCD|       Vanessa|    243.0|
|   BLAH|          John|    250.0|
|    XYZ|         Frank|    340.0|
|     OK|         Sarah|    350.0|
|   BLAH|         Chris|    350.0|
|   ABCD|         Jerry|    600.0|
|   BLAH|          Mike|    750.0|
|     OK|          Carl|    870.0|
+-------+--------------+---------+



In [40]:
df.orderBy(df['num_sales'].desc()).show()

+-------+--------------+---------+
|company|representative|num_sales|
+-------+--------------+---------+
|     OK|          Carl|    870.0|
|   BLAH|          Mike|    750.0|
|   ABCD|         Jerry|    600.0|
|     OK|         Sarah|    350.0|
|   BLAH|         Chris|    350.0|
|    XYZ|         Frank|    340.0|
|   BLAH|          John|    250.0|
|   ABCD|       Vanessa|    243.0|
|    XYZ|           Bob|    200.0|
|   BLAH|         Linda|    130.0|
|   ABCD|           Amy|    124.0|
|    XYZ|           Tom|    120.0|
+-------+--------------+---------+

